# Experiment Analysis Notebook

Loads **metrics dynamically** from each experiment's `metrics.json` / `metrics_*.json`.

Supports: **accuracy, f1_score, precision, recall, roc_auc, loss**

Produces summary tables, comparison charts, and learning curves.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

EXPERIMENTS_DIR = Path('..').resolve()
FIGURES_DIR = EXPERIMENTS_DIR.parent / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

METRIC_COLS = ['accuracy', 'f1_score', 'precision', 'recall', 'roc_auc', 'loss']
print('Experiments dir:', EXPERIMENTS_DIR)
print('Figures dir:', FIGURES_DIR)

## 1. Load all metrics files

In [ ]:
def load_metrics(experiments_dir: Path):
    records = []
    for path in sorted(experiments_dir.glob('*/metrics*.json')):
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        data['_path'] = str(path)
        data['_folder'] = path.parent.name
        records.append(data)
    return records

metrics_list = load_metrics(EXPERIMENTS_DIR)
print(f'Loaded {len(metrics_list)} metrics file(s):')
for m in metrics_list:
    print(f"  - {m.get('experiment_id', m['_folder'])}  source={m.get('source', '?')}")

## 2. Flatten results into a table

In [ ]:
def extract_block(block: dict) -> dict:
    row = {}
    for k in METRIC_COLS:
        row[k] = block.get(k)
    return row

rows = []
for m in metrics_list:
    exp_id = m.get('experiment_id', m['_folder'])
    results = m.get('results', {})

    if 'federated' in results:
        row = {
            'experiment': exp_id,
            'setting': 'federated',
            'source': m.get('source'),
            **extract_block(results['federated']),
        }
        rows.append(row)

    if 'local' in results:
        row = {
            'experiment': exp_id,
            'setting': f"local_{m.get('client_id', 'client')}",
            'source': m.get('source'),
            **extract_block(results['local']),
        }
        rows.append(row)

    if 'centralized' in results:
        row = {
            'experiment': exp_id,
            'setting': 'centralized',
            'source': m.get('source'),
            **extract_block(results['centralized']),
        }
        rows.append(row)

    for key in (
        'local_client_1_express', 'local_client_2_standard',
        'local_client_1_europe', 'local_client_2_latam',
    ):
        if key in results:
            row = {
                'experiment': exp_id,
                'setting': key,
                'source': m.get('source'),
                **extract_block(results[key]),
            }
            rows.append(row)

df = pd.DataFrame(rows)
df

## 3. Main comparison chart (Accuracy, F1, ROC-AUC)

Uses preferred experiment rows when present; falls back to whatever is loaded.

In [ ]:
preferred = [
    ('05_neural_network_local_baselines', 'local_client_1_express', 'Local C1 Express'),
    ('05_neural_network_local_baselines', 'local_client_2_standard', 'Local C2 Standard'),
    ('02_logistic_regression_strong_non_iid', 'federated', 'Federated LR'),
    ('04_neural_network_3rounds', 'federated', 'Federated NN'),
    ('06_centralized_neural_network', 'centralized', 'Centralized NN'),
]

labels, accs, f1s, aucs = [], [], [], []

for exp, setting, label in preferred:
    subset = df[(df['experiment'] == exp) & (df['setting'] == setting)]
    if subset.empty and setting.startswith('local'):
        # match client_id-style settings from new exports
        if '1' in setting or 'express' in setting:
            subset = df[df['setting'].str.contains('client_1|express', case=False, na=False)]
        else:
            subset = df[df['setting'].str.contains('client_2|standard', case=False, na=False)]
        subset = subset[subset['experiment'].str.contains('05|local', case=False, na=False)]
    if not subset.empty and pd.notna(subset.iloc[-1].get('accuracy')):
        row = subset.iloc[-1]
        labels.append(label)
        accs.append(row['accuracy'])
        f1s.append(row['f1_score'] if pd.notna(row.get('f1_score')) else 0.0)
        aucs.append(row['roc_auc'] if pd.notna(row.get('roc_auc')) else np.nan)

if len(labels) < 2:
    labels, accs, f1s, aucs = [], [], [], []
    for _, row in df.dropna(subset=['accuracy']).iterrows():
        labels.append(f"{str(row['experiment'])[:18]}|{row['setting']}")
        accs.append(row['accuracy'])
        f1s.append(row['f1_score'] if pd.notna(row.get('f1_score')) else 0.0)
        aucs.append(row['roc_auc'] if pd.notna(row.get('roc_auc')) else np.nan)

x = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(x - width, accs, width, label='Accuracy', color='#4C72B0')
b2 = ax.bar(x, f1s, width, label='F1', color='#55A868')
# ROC-AUC only if at least one value present
if np.any(pd.notna(aucs)):
    auc_plot = [0.0 if pd.isna(v) else v for v in aucs]
    b3 = ax.bar(x + width, auc_plot, width, label='ROC-AUC', color='#C44E52')
    ax.bar_label(b3, fmt='%.3f', padding=3, fontsize=7)

ax.set_ylabel('Score')
ax.set_title('Performance Comparison (from metrics.json)')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha='right')
ax.set_ylim(0, 1.05)
ax.legend()
ax.bar_label(b1, fmt='%.3f', padding=3, fontsize=7)
ax.bar_label(b2, fmt='%.3f', padding=3, fontsize=7)
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
out = FIGURES_DIR / 'comparison_from_metrics.png'
plt.savefig(out, dpi=300)
plt.show()
print('Saved:', out)

## 4. Precision / Recall comparison

In [ ]:
prec_vals, rec_vals, pr_labels = [], [], []
for _, row in df.iterrows():
    if pd.notna(row.get('precision')) or pd.notna(row.get('recall')):
        pr_labels.append(f"{str(row['experiment'])[:16]}|{row['setting']}")
        prec_vals.append(row['precision'] if pd.notna(row.get('precision')) else 0.0)
        rec_vals.append(row['recall'] if pd.notna(row.get('recall')) else 0.0)

if not pr_labels:
    print('No precision/recall found yet. Re-run experiments with the updated metric export.')
else:
    x = np.arange(len(pr_labels))
    width = 0.35
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x - width/2, prec_vals, width, label='Precision', color='#8172B3')
    ax.bar(x + width/2, rec_vals, width, label='Recall', color='#CCB974')
    ax.set_xticks(x)
    ax.set_xticklabels(pr_labels, rotation=20, ha='right', fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Precision and Recall (from metrics.json)')
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    out = FIGURES_DIR / 'precision_recall_from_metrics.png'
    plt.savefig(out, dpi=300)
    plt.show()
    print('Saved:', out)

## 5. Learning curves (per-round federated metrics)

In [ ]:
def extract_per_round(m):
    fed = m.get('results', {}).get('federated', {})
    pr = fed.get('per_round', {})
    if not pr:
        return None
    return {
        'id': m.get('experiment_id', m['_folder']),
        'accuracy': pr.get('accuracy', []),
        'f1_score': pr.get('f1_score', []),
        'precision': pr.get('precision', []),
        'recall': pr.get('recall', []),
        'roc_auc': pr.get('roc_auc', []),
        'loss': pr.get('loss', []),
    }

curves = [c for c in (extract_per_round(m) for m in metrics_list) if c and (c['accuracy'] or c['loss'])]

if not curves:
    print('No per-round metrics found.')
else:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.ravel()
    series_keys = ['loss', 'accuracy', 'f1_score', 'precision', 'recall', 'roc_auc']
    titles = ['Loss', 'Accuracy', 'F1', 'Precision', 'Recall', 'ROC-AUC']

    for ax, key, title in zip(axes, series_keys, titles):
        plotted = False
        for c in curves:
            vals = c.get(key) or []
            if vals:
                ax.plot(range(1, len(vals) + 1), vals, marker='o', label=c['id'][:24])
                plotted = True
        ax.set_title(title)
        ax.set_xlabel('Round')
        ax.grid(True, linestyle='--', alpha=0.7)
        if plotted:
            ax.legend(fontsize=6)
        else:
            ax.text(0.5, 0.5, 'no data', ha='center', va='center', transform=ax.transAxes)

    plt.suptitle('Learning Curves (from metrics.json)', fontsize=13)
    plt.tight_layout()
    out = FIGURES_DIR / 'learning_curves_from_metrics.png'
    plt.savefig(out, dpi=300)
    plt.show()
    print('Saved:', out)

## 6. Export summary CSV

In [ ]:
summary_path = FIGURES_DIR / 'metrics_summary.csv'
df.to_csv(summary_path, index=False)
print('Saved:', summary_path)
df